# PadelVision — 07 Highlights (Kaggle)

Cuts the best rallies of a match into one video, plus a condensed match with every rally. Each clip has a title
card, the ball's path, hit/bounce markers, the mini court and the stats overlay, and is followed by a 3 s summary
card (each player's heatmap in that rally and its shot map: hitter -> where the ball landed). No model is needed
for the highlights themselves: the clips come from the detected rallies.

**Inputs:** the video, and a finished run so nothing is detected again:
- the `events_review.zip` downloaded from notebook 06, uploaded as a Kaggle Dataset (attach it here), or
- `/kaggle/working/runs/match1` if notebook 06 ran in this same session.

The light stages (players, ball tracking, events, rally stats) are refreshed from the cached detections in about a
minute, so the latest code and calibration always apply. **No GPU needed** with a cached run. Without one, the full
pipeline runs first (GPU T4, ~10 min).

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: fine if a cached run is attached"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    # Skip pipeline outputs (previews, highlights) that sit inside an attached run folder.
    videos = [p for p in videos if not (p.parent / "rallies.json").exists()
              and not p.stem.startswith(("rally_preview", "preview", "highlights", "condensed"))]
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# court.json: pasted below, else ml/calibrations/<video name>.json in the repo, else court.json next to the video.
COURT_JSON_TEXT = """
"""

court_files = sorted(Path("/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/padel").rglob("court.json"))
# Calibrations committed to the repo, named after the video file (e.g. ml/calibrations/Test_video.json).
repo_cal = Path(REPO) / "ml" / "calibrations" / f"{Path(VIDEO_PATH).stem}.json"
if repo_cal.exists():
    court_files.insert(0, repo_cal)
COURT_JSON = f"{WORK}/court.json"
if COURT_JSON_TEXT.strip():
    Path(COURT_JSON).write_text(COURT_JSON_TEXT)
elif court_files:
    Path(COURT_JSON).write_text(court_files[0].read_text())
    print("Using", court_files[0])
else:
    raise FileNotFoundError("No court.json: attach it to the dataset or paste it above")

from padelvision.court import CourtCalibration
cal = CourtCalibration.load(COURT_JSON)
print(f"court: {cal.accuracy_summary()}; lens k1={cal.lens.k1:.3f} k2={cal.lens.k2:.3f}")

## 1. Run folder (cached detections, refreshed stages)

In [ ]:
import json
import shutil

# Leave empty to search the attached datasets and /kaggle/working for a run with cached detections.
RUN_SOURCE = ""
NEEDED = ["video.json", "stats.json", "detections.parquet", "detections.json",
          "ball_detections.parquet", "ball_detections.json"]

def is_run(d: Path) -> bool:
    return all((d / n).exists() for n in NEEDED)

roots = [Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel"), Path(WORK)]
found = [Path(RUN_SOURCE)] if RUN_SOURCE else sorted(
    {p.parent for root in roots if root.exists() for p in root.rglob("ball_detections.parquet")})
found = [d for d in found if is_run(d)]
print("Cached runs found:", *found, sep="\n  ")

# A cached run must come from THIS video: otherwise its rally times would cut the wrong moments.
from padelvision.video import probe
info = probe(VIDEO_PATH)

def same_video(d: Path) -> bool:
    v = json.loads((d / "video.json").read_text())
    ok = (v["width"], v["height"], v["frame_count"]) == (info.width, info.height, info.frame_count)
    ok = ok and abs(v["fps"] - info.fps) < 0.01
    if not ok:
        print(f"  skipping {d}: made from {Path(v.get('path', '?')).name} "
              f"({v['width']}x{v['height']}, {v['frame_count']} frames @ {v['fps']:.2f} fps), "
              f"not this video ({info.width}x{info.height}, {info.frame_count} frames @ {info.fps:.2f} fps)")
    return ok

found = [d for d in found if same_video(d)]
print("Cached runs of this video:", *found, sep="\n  ")

RUN_DIR = f"{WORK}/runs/highlights"
if found:
    if Path(RUN_DIR).resolve() != found[0].resolve():
        shutil.rmtree(RUN_DIR, ignore_errors=True)
        shutil.copytree(found[0], RUN_DIR)  # attached datasets are read-only
    !cd {REPO} && python -m padelvision.cli restats {RUN_DIR} --court {COURT_JSON}
    !cd {REPO} && python -m padelvision.cli ball-retrack {RUN_DIR}
else:
    print("No cached run: running the full pipeline (needs a GPU, ~10 min).")
    shutil.rmtree(RUN_DIR, ignore_errors=True)  # never reuse another video's cached detections
    !cd {REPO} && python -m padelvision.cli analyze "{VIDEO_PATH}" --court {COURT_JSON} --out {RUN_DIR}
    !cd {REPO} && python -m padelvision.cli ball-track "{VIDEO_PATH}" --run {RUN_DIR}
!cd {REPO} && python -m padelvision.cli events {RUN_DIR}

## 2. Highlights

`TOP_N` best rallies by score (hits + length + fastest measured shot), shown in match order, and optionally the
condensed match (every rally). Clip boundaries are only as good as hit detection: a missed hit can split a point
or start a clip late.

In [ ]:
import json
from IPython.display import Video, display

TOP_N = 5
MAKE_CONDENSED = True

modes = [("top", "highlights")] + ([("all", "condensed")] if MAKE_CONDENSED else [])
for mode, name in modes:
    !cd {REPO} && python -m padelvision.cli highlights "{VIDEO_PATH}" --run {RUN_DIR} --mode {mode} --top {TOP_N} --out {RUN_DIR}/{name}_raw.mp4
    raw = Path(f"{RUN_DIR}/{name}_raw.mp4")
    if raw.exists():
        !ffmpeg -loglevel error -y -i {raw} -c:v libx264 -pix_fmt yuv420p -crf 24 {RUN_DIR}/{name}.mp4 && rm {raw}

hl = json.loads(Path(f"{RUN_DIR}/highlights.json").read_text())
print(f"{len(hl['clips'])} clips, {hl['total_s']} s")
for c in hl["clips"]:
    print(f"  rally {c['rally_id']:>3}  {c['start_s']:7.1f}-{c['end_s']:7.1f} s  hits {c['hits']:>2}  "
          f"fastest {c['max_shot_kmh']}  score {c['score']}")
if Path(f"{RUN_DIR}/highlights.mp4").exists():
    display(Video(f"{RUN_DIR}/highlights.mp4", embed=True, width=960))

## 3. Download

In [ ]:
from IPython.display import FileLink

OUT = Path(f"{WORK}/highlights_out")
shutil.rmtree(OUT, ignore_errors=True)
OUT.mkdir(parents=True)
for name in ["highlights.mp4", "highlights.json", "condensed.mp4", "condensed.json", "rallies.json",
             "events.parquet", "stats.json", "placement.png", "heatmaps.png"]:
    p = Path(RUN_DIR) / name
    if p.exists():
        shutil.copy(p, OUT / name)
ZIP = shutil.make_archive(f"{WORK}/highlights", "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)